### File Tools

In [ ]:
### Import Packages ###
import os, csv, glob, aiofiles.os, asyncio, time
import pandas as pd
import tkinter as tk
from datetime import datetime
from tkinter.filedialog import askopenfilename, askdirectory
print(">>> loaded")

#### Get File Names from a Folder Directory

In [ ]:
folder_empty = True
while folder_empty:
    folder_to_search = askdirectory(title = "Select Folder")
    print(f"> Selected folder: {folder_to_search}\n")

    folder_empty = len(os.listdir(folder_to_search)) == 0
    if folder_empty:
        print("> Selected folder is empty, please select a folder that contains files.")

log_file = folder_to_search + "_file_names.txt"

# Find the folder to be searched
for root, dirs, files in os.walk(folder_to_search):
    if folder_to_search in root:
        os.chdir(root)
        os.chdir('..')

        # Create a .txt file to log all the file names in
        logfile = open(log_file, "a", encoding = 'utf8')

        print(f"> Getting all file names from folder [{root}]...\n")
        for f in files:
            logfile.write(f + "\n")
        print("> done\n")
        
        break

print(f">>> File names in folder [{folder_to_search}] are logged in [{log_file}]\n")
logfile.close()

#### Delete Files Last Modified Past a Specific Number of Days

In [ ]:
folder_empty = True
while folder_empty:
    folder_to_search = askdirectory(title = "Select Folder")
    print(f"> Selected folder: {folder_to_search}\n")

    folder_empty = len(os.listdir(folder_to_search)) == 0
    if folder_empty:
        print("> Selected folder is empty, please select a folder that contains files.")

print("> Please enter a whole number, which represents the threshold for the number of days.")
print("> Note that files that were last modified for more than this number of days old will be deleted.\n")

input_check = True
while input_check == True:
    number_of_days = input("> Number of days: ")
    try:
        number_of_days = int(number_of_days)
        input_check = False
    except:
        print("> Input must be a number, please try again.\n")

print(f"> Number of days entered: {number_of_days}\n")
print(f"> Deleting files that are more than {number_of_days} days old from [{folder_to_search}]...")

async def clean_up(folder_path, number_of_days):
    now = time.time()
    cutoff_time = now - (number_of_days * 86400)

    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)

        if os.path.getmtime(file_path) < cutoff_time:
            await aiofiles.os.remove(file_path)
            print(f"> Deleted [{filename}]")
    
    print("> done\n")

asyncio.run(clean_up(folder_to_search, number_of_days))

#### Rename Files

In [ ]:
curr_dir = os.getcwd()
print(f">>> Current Directory: '{curr_dir}'")

target_dir = curr_dir + '/file-tools/target_folder/'
print(f">>> Target Directory: '{target_dir}'")

In [ ]:
# Set the prefix
old = "apple"
new = "orange"

def rename(target_dir, old_string, new_string):
    for filename in os.listdir(target_dir):
        if old_string in filename:
            new_filename = filename.replace(old_string, new_string)
            os.rename(os.path.join(target_dir, filename), os.path.join(target_dir, new_filename))
            print(f"Renamed '{filename}' to '{new_filename}'")

rename(target_dir, old, new)

#### Rename Files with Mapping File

In [ ]:
# Find and read the mapping .csv file
curr_dir = os.getcwd()
csv_mapping_file = "rename-map"
target_folder = "raw files"

for root, dirs, files in os.walk(curr_dir):
    for i in files:
        if csv_mapping_file in i:
            inputFileDir = os.path.join(root, i)
            data1 = pd.read_csv(os.path.join(root, i))

# Get required information from .csv file
newNameList   = data1["Title"]
extensionList = data1["FileExtension"]

# Rename the files
for root, dirs, files in os.walk(os.getcwd() + "/" + target_folder):
    for fileName in files:
        index = data1[data1["Id"] == fileName].index.values[0]
        newFileName = newNameList[index] + "." + extensionList[index]

        os.rename( os.path.join(root, fileName), os.path.join(root, newFileName) )